In [7]:
import rl_utils as rl_utils

test_time = "0327-0940"
checkpoints_path = '/home/ubuntu/ws/checkpoints/'+test_time
restore_from = 1000
buffer_size = 20000

replay_buffer = rl_utils.ReplayBuffer(buffer_size)
replay_buffer.load(f"{checkpoints_path}/{restore_from}_buffer.pth")  

In [9]:
replay_buffer.print_buffer()

Experience 0: (array([ 0.02530218, -0.39908388, -0.21215509], dtype=float32), array([-1.  ,  0.68,  0.82]), -0.370874305434662, array([ 0.02921188, -0.39912087, -0.21742137], dtype=float32), False)
Experience 1: (array([ 0.02921188, -0.39912087, -0.21742137], dtype=float32), array([-0.95,  0.88,  1.  ]), -0.3850845642323688, array([ 0.03313586, -0.39943644, -0.22258529], dtype=float32), False)
Experience 2: (array([ 0.03313586, -0.39943644, -0.22258529], dtype=float32), array([-0.55,  0.53,  1.  ]), -0.42083331193237883, array([ 0.03708469, -0.40005147, -0.22763787], dtype=float32), False)
Experience 3: (array([ 0.03708469, -0.40005147, -0.22763787], dtype=float32), array([-0.76,  0.76,  0.73]), -0.43009796794401767, array([ 0.04170643, -0.4011582 , -0.23356634], dtype=float32), False)
Experience 4: (array([ 0.04170643, -0.4011582 , -0.23356634], dtype=float32), array([-1.  ,  0.71,  1.  ]), -0.3798292596714221, array([ 0.04605616, -0.40249884, -0.23889713], dtype=float32), False)
Expe

In [12]:
import numpy as np

def cal_reward(v_action, v_distance, dt=0.05, alpha=0.8, beta=5.0, w_angle=0.3, w_speed=0.2, w_dist=0.5):
        """
        v_action: 三维速度向量 [vx, vy, vz]
        v_distance: 三维相对位置 [dx, dy, dz]

        dt : float - 时间步长(默认0.05s)
        alpha : float - 动态速度比例系数(默认0.5)
        beta : float - 速度奖励衰减系数(默认10.0)
        w_angle : float - 方向奖励权重(默认0.3)
        w_speed : float - 速度奖励权重(默认0.2)     
        w_dist : float - 距离奖励权重(默认0.5)   
        """

        # 计算物理约束范围
        act_max = np.sqrt(3)                   # 速度模长最大值：√3 ≈ 1.732
        dis_max = np.sqrt(3)  # 动作空间范围假设为 [-1, 1]，则速度模长最大为 √3
        
        # 归一化处理（防止除以零）
        norm_dis = np.linalg.norm(v_distance) / (dis_max + 1e-8)  # 目标距离归一化到[0,1]
        norm_act = np.linalg.norm(v_action) / (act_max + 1e-8)  # 速度模长归一化到[0,1]

        # 方向奖励（余弦相似度）
        dot_product = np.dot(v_action, v_distance)
        norm_act_denominator = np.linalg.norm(v_action) + 1e-8  # 防止除零
        norm_dis_denominator = np.linalg.norm(v_distance) + 1e-8
        cos_sim = dot_product / (norm_act_denominator * norm_dis_denominator)

        # 动态理想速度模型（方案一）
        # v_ideal = min(alpha * norm_dis, 1.0)  # 限制归一化后的理想速度不超过1
        v_ideal = alpha * norm_dis
        speed_diff = norm_act - v_ideal
        speed_reward = np.exp(-beta * (speed_diff ** 2))  # 高斯型速度奖励

        # 距离奖励
        distance_reward = -1 * norm_dis

        # 综合奖励
        return w_angle * cos_sim + w_speed * speed_reward * np.sign(cos_sim) + w_dist * distance_reward

In [ ]:
def normalize_state(actual_diff):
    """将实际坐标差值转换为归一化值
    Args:
        actual_diff: numpy数组，形状(3,) 表示 [dx, dy, dz]
    Returns:
        归一化后的numpy数组，形状(3,)
    """
    return np.array([
        actual_diff[0] / (2 * 5),
        actual_diff[1] / (2 * 5),
        actual_diff[2] / 10
    ], dtype=np.float32)

In [14]:
v_a = [ 0.64, 0.23, 0.33]
v_d = [ -3.93, -4.82, -5.23]
reward = cal_reward(v_a, v_d)
print(f"Reward: {reward:.4f}")

Reward: -2.6070
